# Capstone 1 — Safe Expression Calculator

A small calculator that evaluates arithmetic expressions without calling `eval`. The parser is a recursive-descent parser over a token stream. It supports the four operations, floor division, remainder, power, parentheses, unary minus, a memory register, and a history of successful evaluations.

This is the right shape for a first capstone: a clear grammar, a class boundary, and failure cases that must be reported rather than ignored.

## Grammar

```text
expression = term   { (+|-) term }
term       = power  { (*|/|//|%) power }
power      = unary  [ ** power ]          # right associative
unary      = - unary | primary
primary    = number | ( expression ) | memory
```

Power is right associative, so `2 ** 3 ** 2` is `2 ** (3 ** 2)`, matching Python. Division by zero and a trailing operator are errors, not crashes.


In [1]:
from dataclasses import dataclass

class CalculatorError(Exception):
    """Raised when an expression cannot be evaluated."""


@dataclass(frozen=True)
class Token:
    kind: str
    value: str


class Lexer:
    def __init__(self, source: str):
        self.source = source
        self.index = 0

    def tokens(self) -> list[Token]:
        produced = []
        while self.index < len(self.source):
            char = self.source[self.index]
            if char.isspace():
                self.index += 1
                continue
            if char.isdigit() or char == ".":
                produced.append(self._number())
                continue
            if self.source.startswith("//", self.index):
                produced.append(Token("OP", "//"))
                self.index += 2
                continue
            if self.source.startswith("**", self.index):
                produced.append(Token("OP", "**"))
                self.index += 2
                continue
            if char in "+-*/%()":
                kind = "LPAREN" if char == "(" else "RPAREN" if char == ")" else "OP"
                produced.append(Token(kind, char))
                self.index += 1
                continue
            if char == "M":
                produced.append(Token("MEMORY", "M"))
                self.index += 1
                continue
            raise CalculatorError(f"unexpected character: {char!r}")
        produced.append(Token("EOF", ""))
        return produced

    def _number(self) -> Token:
        start = self.index
        seen_dot = False
        while self.index < len(self.source):
            char = self.source[self.index]
            if char.isdigit():
                self.index += 1
            elif char == "." and not seen_dot:
                seen_dot = True
                self.index += 1
            else:
                break
        literal = self.source[start:self.index]
        if literal == ".":
            raise CalculatorError("incomplete number")
        return Token("NUMBER", literal)


class Parser:
    def __init__(self, tokens: list[Token], memory: float):
        self.tokens = tokens
        self.memory = memory
        self.index = 0

    def parse(self) -> float:
        value = self._expression()
        self._expect("EOF")
        return value

    def _expression(self) -> float:
        value = self._term()
        while self._peek().value in {"+", "-"} and self._peek().kind == "OP":
            op = self._advance().value
            right = self._term()
            value = value + right if op == "+" else value - right
        return value

    def _term(self) -> float:
        value = self._power()
        while self._peek().value in {"*", "/", "//", "%"}:
            op = self._advance().value
            right = self._power()
            if op in {"/", "//", "%"} and right == 0:
                raise CalculatorError("division by zero")
            if op == "*":
                value *= right
            elif op == "/":
                value /= right
            elif op == "//":
                value = value // right
            else:
                value %= right
        return value

    def _power(self) -> float:
        value = self._unary()
        if self._peek().value == "**":
            self._advance()
            return value ** self._power()
        return value

    def _unary(self) -> float:
        if self._peek().kind == "OP" and self._peek().value == "-":
            self._advance()
            return -self._unary()
        return self._primary()

    def _primary(self) -> float:
        token = self._advance()
        if token.kind == "NUMBER":
            return float(token.value)
        if token.kind == "MEMORY":
            return self.memory
        if token.kind == "LPAREN":
            value = self._expression()
            self._expect("RPAREN")
            return value
        raise CalculatorError(f"expected a value, found {token.value or token.kind!r}")

    def _peek(self) -> Token:
        return self.tokens[self.index]

    def _advance(self) -> Token:
        token = self._peek()
        self.index += 1
        return token

    def _expect(self, kind: str) -> Token:
        token = self._advance()
        if token.kind != kind:
            raise CalculatorError(f"expected {kind}, found {token.value or token.kind!r}")
        return token


class Calculator:
    """Expression calculator with memory and a bounded history."""

    def __init__(self):
        self.memory = 0.0
        self.history: list[tuple[str, float]] = []

    def evaluate(self, expression: str) -> float:
        tokens = Lexer(expression).tokens()
        value = Parser(tokens, self.memory).parse()
        self.history.append((expression, value))
        return value

    def store(self) -> float:
        if not self.history:
            raise CalculatorError("nothing to store")
        self.memory = self.history[-1][1]
        return self.memory

    def clear(self) -> None:
        self.memory = 0.0
        self.history.clear()


calc = Calculator()
samples = ["2 + 3 * 4", "(2 + 3) * 4", "2 ** 3 ** 2", "-5 + 12", "7 // 2", "10 % 4"]
for sample in samples:
    print(f"{sample:>12} = {calc.evaluate(sample)}")
print("stored memory:", calc.store())
print("memory reuse:", calc.evaluate("M * 2 + 1"))
try:
    calc.evaluate("10 / 0")
except CalculatorError as exc:
    print("handled:", exc)
print("history length:", len(calc.history))


   2 + 3 * 4 = 14.0
 (2 + 3) * 4 = 20.0
 2 ** 3 ** 2 = 512.0
     -5 + 12 = 7.0
      7 // 2 = 3.0
      10 % 4 = 2.0
stored memory: 2.0
memory reuse: 5.0
handled: division by zero
history length: 7


## What to notice

Operator precedence is encoded by which method calls which, not by a table consulted at runtime. A lower-precedence rule calls a higher-precedence rule. That is why `2 + 3 * 4` prints `14` rather than `20`.

`M` is a primary, so it can appear anywhere a number can. Memory is updated only by `store`, which keeps evaluation free of hidden side effects.
